# 3. Use the trained anomaly model

This notebook demonstrates inference on one image using the artifact produced by `01_train_model.ipynb`. It does not build a memory bank or calibrate thresholds.

Use `predict.py` for the same workflow from a terminal or another application.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from anomaly_model import (
    ResNetFeatureExtractor,
    create_neighbor_index,
    get_device,
    load_artifact,
    load_config,
    predict,
)

## Choose an image

Set `image_path` to any compatible RGB image. The example below selects the first image from the first defect category.

In [ ]:
# Use the artifact for this category.
TARGET_CATEGORY = 'metal_nut'
config = load_config(Path('config.yaml'), category=TARGET_CATEGORY)
device = get_device()
memory_bank, thresholds, metadata = load_artifact(config['output_dir'] / 'anomaly_model.npz')
model = ResNetFeatureExtractor(metadata['backbone_name']).to(device)
model.eval()
neighbor_index = create_neighbor_index(memory_bank, metadata['num_neighbors'])

defect_dir = next(path for path in (config['dataset_root'] / 'test').iterdir() if path.is_dir() and path.name != 'good')
image_path = sorted(defect_dir.iterdir())[0]
prediction = predict(image_path, model, neighbor_index, thresholds, config, device)
print(f'Target category: {config["category"]}')
print(f'Image: {image_path}')
print(f'Anomaly score: {prediction.anomaly_score:.4f}')
print(f'Detected: {prediction.is_anomalous}')
print(f'Bounding boxes: {prediction.bounding_boxes}')

## Inspect the prediction

The four panels show the source image, predicted binary mask, normalized anomaly map, and localization overlay.

In [ ]:
from PIL import Image

figure, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(Image.open(image_path).convert('RGB'))
axes[0].set_title('Original')
axes[1].imshow(prediction.binary_mask, cmap='gray')
axes[1].set_title('Predicted Mask')
axes[2].imshow(prediction.anomaly_map, cmap='jet', vmin=0, vmax=1)
axes[2].set_title('Anomaly Map')
axes[3].imshow(cv2.cvtColor(prediction.overlay, cv2.COLOR_BGR2RGB))
axes[3].set_title('Localization')
for axis in axes:
    axis.axis('off')
figure.tight_layout()
plt.show()

## Reuse from Python

For application code, import `predict`, load the artifact once at startup, and reuse the model and nearest-neighbor index for each incoming image.